In [0]:
from pyspark.sql import functions as F

CHANGES_TABLE = "workspace.bank_project.customer_changes"

COLUMNS = ["customer_id", "full_name", "city", "risk_rating", "kyc_status", "operation", "change_time", "batch_id"]


def send_batch(batch_id, rows):
    """ Pretend the core banking system sends a bundle of change messages."""

    already_sent = spark.table(CHANGES_TABLE).where(F.col("batch_id") == batch_id ).count()
    if already_sent > 0:
      print(f"Batch{batch_id} was already sent. Nothing new added.")
      return
    df = (spark.createDataFrame(rows,COLUMNS)
          .withColumn("change_time",F.to_timestamp("change_time"))
          .withColumn("batch_id", F.col("batch_id").cast("int")))
    df.write.mode("append").saveAsTable(CHANGES_TABLE)
    print(f"Batch {batch_id} sent with {len(rows)} change message.")

print("Core syatem tool is ready")  

In [0]:
batch_1 = [
    ("C001", "Asha Rao",    "Pune",   "Low",    "VERIFIED", "INSERT", "2024-01-01 09:00:00", 1),
    ("C002", "Rahul Mehta", "Mumbai", "MEDIUM", "VERIFIED", "INSERT", "2024-01-01 09:00:00", 1),
    ("C003", "Priya Singh", "Delhi",  "Low",    "PENDING",  "INSERT", "2024-01-01 09:00:00", 1),
    ("C004", "Vikram Das",  "Kolkata","HIGH",   "VERIFIED", "INSERT", "2024-01-01 09:00:00", 1),
    ("C005", "Neha Iyer",   "Chennai","LOW",    "VERIFIED", "INSERT", "2024-01-01 09:00:00", 1)
]
send_batch(1, batch_1)

In [0]:
batch_2 = [
("C001", "Asha Rao", "Bengaluru", "LOW", "VERIFIED", "UPDATE", "2024-03-15 10:00:00", 2),
("C003", "Priya Singh", "Delhi", "LOW", "VERIFIED", "UPDATE", "2024-03-15 11:00:00", 2),
("C004", "Vikram Das", "Kolkata", "MEDIUM", "VERIFIED", "UPDATE", "2024-03-15 12:00:00", 2),
("C006", "Arjun Nair", "Kochi", "LOW", "PENDING", "INSERT", "2024-03-15 13:00:00", 2)
]
send_batch(2, batch_2)

In [0]:
display(spark.sql("""
                  SELECT * FROM workspace.bank_project.customer_changes
                  ORDER BY batch_id, customer_id
                  """))

In [0]:
batch_3 = [
# Normal change: Asha's risk rating goes up
("C001", "Asha Rao", "Bengaluru", "HIGH", "VERIFIED", "UPDATE", "2024-06-01 09:00:00", 3),
# Account closed: Rahul leaves the bank
("C002", None, None, None, None, "DELETE", "2024-06-01 10:00:00", 3),
# LATE message: this really happened on 20 Feb, but only arrives now
("C003", "Priya Singh", "Delhi", "LOW", "UNDER_REVIEW", "UPDATE", "2024-02-20 14:00:00", 3),
]
send_batch(3, batch_3)